# Discover Portugal — Notebook 01: Data Cleaning

In [410]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
from sklearn.neighbors import KNeighborsClassifier

# Folders
RAW = "../data/raw/"
CLEAN = "../data/clean/"

# Show all columns when we print a table
pd.set_option("display.max_columns", None)

### Helper functions

In [411]:
def simple_name(text):
    """'Évora ' -> 'evora' (no accents, lowercase, no spaces at the ends)"""
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode()
    return text.lower().strip()


def distance_km(lat1, lon1, lat2, lon2):
    """Distance in km between point 1 and point 2 (works with numbers or arrays)"""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

---
## 1. Overnight stays by month (INE)

This file answers **Q1 – Where do tourists go?** and **Q2 – When?**

### 1.1 Load the file

In [412]:
# geo_code must be text: some codes have letters (e.g. "11D0404")
month_raw = pd.read_csv(RAW + "ine_overnight_stays_region_month.csv", dtype={"geo_code": str})

print(month_raw.shape)
month_raw.head(20)

(60544, 8)


,month_code,month,geo_code,geo_name,segment_code,segment,overnight_stays,flag
0,S3A202301,January 2023,300,Região Autónoma da Madeira,01,Hotels and similar,520254.0,NaN
1,S3A202301,January 2023,3003101,Calheta,01,Hotels and similar,17213.0,NaN
2,S3A202301,January 2023,3003102,Câmara de Lobos,01,Hotels and similar,8592.0,NaN
3,S3A202301,January 2023,3003103,Funchal,01,Hotels and similar,387154.0,NaN
4,S3A202301,January 2023,3003104,Machico,01,Hotels and similar,10081.0,NaN
5,S3A202301,January 2023,3003105,Ponta do Sol,01,Hotels and similar,6799.0,NaN
6,S3A202301,January 2023,3003106,Porto Moniz,01,Hotels and similar,5653.0,NaN
7,S3A202301,January 2023,3003107,Ribeira Brava,01,Hotels and similar,NaN,Confidential data
8,S3A202301,January 2023,3003108,Santa Cruz,01,Hotels and similar,64494.0,NaN
9,S3A202301,January 2023,3003109,Santana,01,Hotels and similar,NaN,Confidential data


### 1.2 The geography has different levels

The file mixes **country, regions and municipalities** in the same column.
The length of `geo_code` tells us the level:

| Length of geo_code | Level | Example |
|---|---|---|
| 2 (`PT`) | Country | Portugal |
| 1 | Mainland / islands | Continente |
| 2 | Region (NUTS II) | Algarve |
| 3 | Sub-region (NUTS III) | Alentejo Central |
| 7 | **Municipality** (concelho) | Évora |

⚠️ If we sum everything we count the same tourists 3 or 4 times! So we keep **only the municipalities** and add the region as a new column.

In [413]:
month_raw["code_length"] = month_raw["geo_code"].str.len()
month_raw.groupby("code_length")["geo_code"].nunique()

code_length
1      3
2      8
3     25
7    308
Name: geo_code, dtype: int64

### 1.3 Build a table of municipalities and regions

The first 2 characters of a municipality code are its **region** (NUTS II).

In [414]:
# Short names for the 7 regions (NUTS II, version 2013 used by INE)
REGION_NAMES = {
    "11": "Norte",
    "16": "Centro",
    "17": "Lisboa",
    "18": "Alentejo",
    "15": "Algarve",
    "20": "Açores",
    "30": "Madeira",
}

# One row per municipality
municipalities = month_raw[month_raw["code_length"] == 7][["geo_code", "geo_name"]].drop_duplicates()
municipalities = municipalities.rename(columns={"geo_code": "municipality_code", "geo_name": "municipality"})
municipalities["region"] = municipalities["municipality_code"].str[:2].map(REGION_NAMES)

print(len(municipalities), "municipalities")
municipalities["region"].value_counts()

308 municipalities


region
Centro      100
Norte        86
Alentejo     58
Açores       19
Lisboa       18
Algarve      16
Madeira      11
Name: count, dtype: int64

### 1.4 Two names appear twice

There are two **Calheta** (Madeira and Açores) and two **Lagoa** (Algarve and Açores).
We add the region to the name so every municipality has a unique name.

In [415]:
municipalities["municipality"].value_counts()

municipality
Calheta                 2
Lagoa                   2
Câmara de Lobos         1
Funchal                 1
Machico                 1
                       ..
Valpaços                1
Vila Pouca de Aguiar    1
Boticas                 1
Castelo de Paiva        1
Celorico de Basto       1
Name: count, Length: 306, dtype: int64

In [416]:
print(municipalities[municipalities["municipality"].duplicated(keep=False)])

duplicated = municipalities["municipality"].duplicated(keep=False)
municipalities.loc[duplicated, "municipality"] = (
    municipalities.loc[duplicated, "municipality"] + " (" + municipalities.loc[duplicated, "region"] + ")"
)

municipalities[municipalities["municipality"].str.contains("Calheta|Lagoa")]

    municipality_code municipality   region
1             3003101      Calheta  Madeira
49            2004501      Calheta   Açores
59            2004201        Lagoa   Açores
303           1500806        Lagoa  Algarve


,municipality_code,municipality,region
1,3003101,Calheta (Madeira),Madeira
49,2004501,Calheta (Açores),Açores
59,2004201,Lagoa (Açores),Açores
303,1500806,Lagoa (Algarve),Algarve


### 1.5 Missing values

Some numbers are empty. The `flag` column tells us why:
- **Confidential data** → INE hides the number (very few hotels, so it could identify them)
- **Null or not applicable** → there is nothing to count

💡 We keep them empty (`NaN`). When we sum, pandas ignores them.

In [417]:
month_raw["flag"].value_counts(dropna=False)

flag
NaN                       37515
Confidential data         17972
Null or not applicable     5057
Name: count, dtype: int64

### 1.6 Clean table: municipalities only, with region, year and month

INE also splits the stays by **type of accommodation** (hotels, local accommodation, rural tourism). We only need the **Total**.

In [418]:
stays_month = month_raw[month_raw["code_length"] == 7].copy()
stays_month = stays_month.rename(columns={"geo_code": "municipality_code"})

# Add municipality name (with the fix for Calheta / Lagoa) and region
stays_month = stays_month.drop(columns=["geo_name"]).merge(municipalities, on="municipality_code", how="left")

# "S3A202301" -> year 2023, month 1
stays_month["year"] = stays_month["month_code"].str[3:7].astype(int)
stays_month["month_number"] = stays_month["month_code"].str[7:9].astype(int)

# Keep only the Total (all types of accommodation together)
stays_month = stays_month[stays_month["segment"] == "Total"]

stays_month = stays_month[["year", "month_number", "region", "municipality_code", "municipality", "overnight_stays"]]
stays_month.head()

,year,month_number,region,municipality_code,municipality,overnight_stays
924,2023,1,Madeira,3003101,Calheta (Madeira),25751.0
925,2023,1,Madeira,3003102,Câmara de Lobos,11058.0
926,2023,1,Madeira,3003103,Funchal,425801.0
927,2023,1,Madeira,3003104,Machico,11669.0
928,2023,1,Madeira,3003105,Ponta do Sol,7940.0


---
## 2. Overnight stays by country of residence (INE)

👉 This file answers **Q3 – Who visits each region?**

In [419]:
nat_raw = pd.read_csv(RAW + "ine_overnight_stays_nationality.csv", dtype={"geo_code": str})
nat_raw["code_length"] = nat_raw["geo_code"].str.len()

print(nat_raw.shape)
print(nat_raw["year"].value_counts().sort_index())
nat_raw.head()

(86000, 8)
year
2019    17200
2022    17200
2023    17200
2024    17200
2025    17200
Name: count, dtype: int64


,year,geo_code,geo_name,country_code,country,overnight_stays,flag,code_length
0,2019,1111602,Caminha,TR,Türkiye,1.0,NaN,7
1,2019,1111603,Melgaço,IN,India,1.0,NaN,7
2,2019,1111603,Melgaço,UA,Ukraine,1.0,NaN,7
3,2019,1111604,Monção,GR,Greece,1.0,NaN,7
4,2019,1111604,Monção,KR,Korea (the Republic of),1.0,NaN,7


### 2.1 Keep municipalities and shorten the country names

Some country names are very long (*"United Kingdom of Great Britain and Northern Ireland (the)"*). We make them short for the charts.

The file also has 3 special rows that are **not countries**: `Total`, `Portugal` (Portuguese tourists) and `Foreign countries` (all foreigners together). We keep them, they are useful.

In [420]:
SHORT_COUNTRY = {
    "United Kingdom of Great Britain and Northern Ireland (the)": "United Kingdom",
    "United States of America (the)": "United States",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Russian Federation (the)": "Russia",
    "Korea (the Republic of)": "South Korea",
}

stays_nat = nat_raw[nat_raw["code_length"] == 7].copy()
stays_nat = stays_nat.rename(columns={"geo_code": "municipality_code"})
stays_nat = stays_nat.drop(columns=["geo_name"]).merge(municipalities, on="municipality_code", how="left")
stays_nat["country"] = stays_nat["country"].replace(SHORT_COUNTRY)

stays_nat = stays_nat[["year", "region", "municipality_code", "municipality", "country", "overnight_stays"]]
stays_nat.head()

,year,region,municipality_code,municipality,country,overnight_stays
0,2019,Norte,1111602,Caminha,Türkiye,1.0
1,2019,Norte,1111603,Melgaço,India,1.0
2,2019,Norte,1111603,Melgaço,Ukraine,1.0
3,2019,Norte,1111604,Monção,Greece,1.0
4,2019,Norte,1111604,Monção,South Korea,1.0


### 2.2 Quick check: does the sum of the municipalities match Portugal?

If the cleaning is right, the sum of all municipalities should be (almost) the number for Portugal.

In [421]:
pt_total = nat_raw[(nat_raw["geo_code"] == "PT") & (nat_raw["country"] == "Total")].set_index("year")["overnight_stays"]
mun_total = stays_nat[stays_nat["country"] == "Total"].groupby("year")["overnight_stays"].sum()

check = pd.DataFrame({"Portugal (INE)": pt_total, "Sum of municipalities": mun_total})
check["difference %"] = (1 - check["Sum of municipalities"] / check["Portugal (INE)"]) * 100
check.round(1)

,Portugal (INE),Sum of municipalities,difference %
year,,,
2019,70158964.0,70101007.0,0.1
2022,69694791.0,69694791.0,0.0
2023,77179150.0,77179150.0,0.0
2024,80354833.0,80354833.0,0.0
2025,82011658.0,82011658.0,0.0


💡 The small difference comes from the confidential values that INE hides.

---
## 3. The places

### 3.1 A function to find the INE municipality name

- We compare names without accents (`simple_name`)
- A few names are written in a different way in other sources, so we fix them by hand
- For **Calheta** and **Lagoa** we use the longitude: the Açores are far to the west (longitude < -20)

In [422]:
# simple name -> INE name
INE_NAMES = dict(zip(municipalities["municipality"].map(simple_name), municipalities["municipality"]))

# Names that are written differently in the other sources
NAME_FIXES = {
    "freixo espada a cinta": "Freixo de Espada à Cinta",
    "praia da vitoria": "Vila da Praia da Vitória",
    "municipio do corvo": "Corvo",
}


def find_municipality(name, longitude=None):
    """Returns the INE name of a municipality, or None if it is not a municipality"""
    if pd.isna(name):
        return None
    key = simple_name(name)
    if key in NAME_FIXES:
        return NAME_FIXES[key]
    if key in ["calheta", "lagoa"] and longitude is not None and not pd.isna(longitude):
        if longitude < -20:
            return name + " (Açores)"
        return name + (" (Madeira)" if key == "calheta" else " (Algarve)")
    return INE_NAMES.get(key)


# Test
print(find_municipality("evora"), "|", find_municipality("Lagoa", -25.5), "|", find_municipality("Lisbon"))

Évora | Lagoa (Açores) | None


💡 *"Lisbon"* gives `None` because INE writes *"Lisboa"*. We will solve those cases with the coordinates (step 3.7).

### 3.2 🏖️ Blue Flag beaches

The Blue Flag list also has marinas and boats. We keep **only the beaches**.

The type of beach:
- *praia costeira* → **coastal** (sea)
- *praia interior* → **river** (river or lake)
- *praia transição* → **transition** (where the river meets the sea)

In [423]:
blue_flag = pd.read_csv(RAW + "blue_flag_2026.csv")
print(blue_flag["type"].value_counts())

beaches = blue_flag[blue_flag["type"].str.startswith("praia")].copy()
beaches["subcategory"] = beaches["type"].map({
    "praia costeira": "coastal",
    "praia interior": "river",
    "praia transição": "transition",
})
beaches["municipality"] = [find_municipality(m, lon) for m, lon in zip(beaches["municipality"], beaches["longitude"])]

print("\nBeaches:", len(beaches), "| without municipality:", beaches["municipality"].isna().sum())

type
praia costeira                343
praia interior                 45
porto de recreio ou marina     20
embarcação de ecoturismo       12
praia transição                 2
Name: count, dtype: int64

Beaches: 390 | without municipality: 0


### 3.3 🌲 Protected areas (Forest / Nature)

Some protected areas are in **more than one municipality** (e.g. Peneda-Gerês). The names are in one column separated by `|`.

We put all of them in **one category: "natural area"** (no need to separate parks, reserves…).

We use `explode()` to make **one row per municipality**. Some of those names are parishes or regions, not municipalities: `find_municipality` gives `None` for them and we remove them.

In [424]:
protected = pd.read_csv(RAW + "protected_areas_portugal.csv")
print(len(protected), "protected areas")

# All protected areas get the same category
protected["subcategory"] = "natural area"

# One row per municipality
protected["municipality"] = protected["municipalities"].str.split(r" \| ")
protected = protected.explode("municipality").reset_index(drop=True)
protected["municipality"] = [find_municipality(m, lon) for m, lon in zip(protected["municipality"], protected["longitude"])]

# Fix errors in Wikidata
# 1. "Portas de Ródão" is on the mainland, but Wikidata also lists "Santana" (a municipality in Madeira)
protected = protected[~((protected["name"] == "Portas de Ródão") & (protected["municipality"] == "Santana"))]

# 2. Some areas in the Açores have no real coordinates: they all get the same point (in Faial island)
#    Two of them have the island in the name (Flores has 2 municipalities: we use the main one)
protected.loc[protected["name"] == "Nature Park of Santa Maria", "municipality"] = "Vila do Porto"
protected.loc[protected["name"] == "Nature Park of Flores", "municipality"] = "Santa Cruz das Flores"
#    The others (small "fajãs") we can't place on the map -> we remove them
placeholder = (protected["longitude"].round(3) == -28.629) & (protected["latitude"].round(3) == 38.534)
protected = protected[~(placeholder & protected["municipality"].isna())].copy()
#    The 2 parks we kept: we remove the wrong point, so they don't appear in the wrong place on the map
placeholder = (protected["longitude"].round(3) == -28.629) & (protected["latitude"].round(3) == 38.534)
protected.loc[placeholder, ["latitude", "longitude"]] = np.nan
print("Protected areas after the fixes:", len(protected))


304 protected areas
Protected areas after the fixes: 314


### 3.4 🏡 Historic and schist villages (Countryside)

In [425]:
villages = pd.read_csv(RAW + "historic_and_schist_villages.csv")
villages["municipality"] = villages["municipality"].map(find_municipality)

print("Without municipality:", villages["municipality"].isna().sum())
villages.head(20)

Without municipality: 0


,network,village,municipality
0,Aldeias Históricas de Portugal,Almeida,Almeida
1,Aldeias Históricas de Portugal,Castelo Mendo,Almeida
2,Aldeias Históricas de Portugal,Castelo Novo,Fundão
3,Aldeias Históricas de Portugal,Castelo Rodrigo,Figueira de Castelo Rodrigo
4,Aldeias Históricas de Portugal,Idanha-a-Velha,Idanha-a-Nova
5,Aldeias Históricas de Portugal,Linhares da Beira,Celorico da Beira
6,Aldeias Históricas de Portugal,Marialva,Mêda
7,Aldeias Históricas de Portugal,Monsanto,Idanha-a-Nova
8,Aldeias Históricas de Portugal,Piódão,Arganil
9,Aldeias Históricas de Portugal,Sortelha,Sabugal


### 3.5 🏛️ Classified monuments (City / Culture)

The official classifications (from the most to the least important):
- **UNESCO World Heritage** (Património Mundial)
- **Monumento Nacional**
- **Interesse Público** (several types)

In [426]:
heritage = pd.read_csv(RAW + "heritage_monuments_portugal.csv")

def heritage_level(text):
    if "Unesco" in text:
        return "UNESCO World Heritage"
    if "Monumento Nacional" in text:
        return "National Monument"
    return "Public Interest"

heritage["subcategory"] = heritage["designation"].map(heritage_level)

# Some monuments have no name in Wikidata, only the code (e.g. "Q86659235") → remove them
heritage = heritage[~heritage["name"].str.match(r"^Q\d+$")]

# Some monuments have 2 municipalities ("A | B"): we keep the first one
heritage["municipality"] = heritage["municipality"].str.split(r" \| ").str[0]
heritage["municipality"] = [find_municipality(m, lon) for m, lon in zip(heritage["municipality"], heritage["longitude"])]

print(heritage["subcategory"].value_counts())
print("\nWithout municipality:", heritage["municipality"].isna().sum())

subcategory
Public Interest          3179
National Monument         905
UNESCO World Heritage      49
Name: count, dtype: int64

Without municipality: 6


### 3.6 📍 Places with coordinates but without municipality

Museums and viewpoints only have **coordinates**. And some monuments and natural areas have no municipality.

We already have more than 4 000 places where we know the coordinates **and** the municipality (monuments + beaches).
We train a **machine learning model** with them: it learns which municipality each area of the map belongs to.
Then it predicts the municipality of the new places.

We test a few models and keep the one with the best **accuracy** (% of correct municipalities).

In [427]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold

# Places where we know coordinates + municipality (the training data)
known_places = pd.concat([
    heritage[["latitude", "longitude", "municipality"]],
    beaches[["latitude", "longitude", "municipality"]],
]).dropna()

X = known_places[["latitude", "longitude"]]
y = known_places["municipality"]

print("Known places:", len(known_places), "| municipalities:", y.nunique())

Known places: 4494 | municipalities: 299


#### Which model is the best?

We compare 5 models with **cross-validation**:
- the data is split in **5 parts**
- each model is trained 5 times: 4 parts to learn, 1 part to test (a different one each time)
- the score is the **average accuracy** of the 5 tests

This is more reliable than only one train/test split, because the result doesn't depend on which places were chosen for the test.

In [428]:
models = {
    "KNN (K=1)": KNeighborsClassifier(n_neighbors=1),
    "KNN (K=3)": KNeighborsClassifier(n_neighbors=3),
    "KNN (K=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

folds = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=folds, scoring="accuracy")
    results.append({"model": name, "accuracy": scores.mean()})

results = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results["accuracy"] = (results["accuracy"] * 100).round(1)
results

,model,accuracy
0,KNN (K=1),92.8
4,Random Forest,90.3
1,KNN (K=3),90.2
3,Decision Tree,88.1
2,KNN (K=5),87.4


In [429]:
# Keep the best model and train it with ALL the known places
best_name = results.iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X, y)

print("Best model:", best_name)

Best model: KNN (K=1)


💡 The **KNN** wins: with only 2 variables (latitude and longitude), looking for the **closest known place** is the most natural model.
The Random Forest cuts the map in "rectangles", and the borders of the municipalities are curved, so it fits worse.

About 9 in 10 places get the right municipality. The errors are mostly places close to the border between two municipalities. Good enough for our analysis.

In [430]:
def fill_municipality(df):
    """Fill the empty municipalities using the closest known place"""
    missing = df["municipality"].isna() & df["latitude"].notna()
    if missing.any():
        df.loc[missing, "municipality"] = best_model.predict(df.loc[missing, ["latitude", "longitude"]])
    return df

# Monuments without municipality
heritage = fill_municipality(heritage)
# Protected areas where no municipality was found
protected = fill_municipality(protected)
protected = protected.drop_duplicates(subset=["wikidata", "municipality"])

print("Monuments without municipality:", heritage["municipality"].isna().sum())

Monuments without municipality: 2


### 3.7 🖼️ Museums and viewpoints (OpenStreetMap)

In [431]:
osm = pd.read_csv(RAW + "osm_places_portugal.csv")
print(osm["category"].value_counts())

# The OSM beaches are not needed (the Blue Flag list is better) -> keep museums and viewpoints
osm = osm[osm["category"].isin(["museum", "viewpoint"])].copy()

# Remove places without a name (we can't recommend a place without a name)
osm = osm[osm["name"].notna()]

osm["municipality"] = None
osm = fill_municipality(osm)
osm.head()

category
viewpoint    1458
museum       1315
beach        1048
Name: count, dtype: int64


,category,osm_type,osm_id,name,wikidata,latitude,longitude,municipality
0,museum,node,251713990,Casa Típica,NaN,40.201139,-7.639243,Covilhã
1,museum,node,262732396,Museu Paço dos Duques de Bragança,NaN,41.446536,-8.291223,Guimarães
2,museum,node,300363359,Casa Museu Frederico de Freitas,Q54823828,32.649890,-16.912345,Funchal
3,museum,node,300363364,Museu da Quinta das Cruzes,Q20059117,32.651046,-16.913874,Funchal
4,museum,node,320385380,Fábrica da Pólvora de Barcarena,Q10286946,38.740930,-9.284673,Oeiras


### 3.8 All places in one table 🧩

Every source becomes the same columns:

| Column | Meaning |
|---|---|
| `name` | Name of the place |
| `category` | beach / culture / nature / countryside / viewpoint |
| `subcategory` | More detail (coastal, National Monument, natural park…) |
| `quality_marker` | `True` if it has an official quality mark |
| `municipality`, `latitude`, `longitude` | Where it is |

In [432]:
def to_places(df, category, subcategory, quality_marker):
    """Takes one source and returns it with the common columns"""
    out = pd.DataFrame({
        "name": df["name"],
        "category": category,
        "subcategory": subcategory,
        "quality_marker": quality_marker,
        "municipality": df["municipality"],
        "latitude": df["latitude"] if "latitude" in df else np.nan,
        "longitude": df["longitude"] if "longitude" in df else np.nan,
    })
    return out


places = pd.concat([
    to_places(beaches, "beach", beaches["subcategory"], True),
    to_places(heritage, "culture", heritage["subcategory"], True),
    to_places(osm[osm["category"] == "museum"], "culture", "museum", False),
    to_places(protected, "nature", protected["subcategory"], True),
    to_places(villages.rename(columns={"village": "name"}), "countryside", villages["network"], True),
    to_places(osm[osm["category"] == "viewpoint"], "viewpoint", "viewpoint", False),
], ignore_index=True)

# Add the region of each municipality
places = places.merge(municipalities[["municipality", "region"]], on="municipality", how="left")

print(places.shape)
print("Without municipality:", places["municipality"].isna().sum())
places["category"].value_counts()

(7583, 8)
Without municipality: 37


category
culture        5385
viewpoint      1458
beach           390
nature          311
countryside      39
Name: count, dtype: int64

#### Small monuments are not quality markers

Some classified monuments are very small: **road markers** (*marcos*), **pillories** (*pelourinhos*) and **stone crosses** (*cruzeiros*).
Nobody travels to see them, but some municipalities have many, and they would look like "hidden gems" without really being one.
So they stay in our list of places, but they **don't count as quality markers**.

In [433]:
# Names that start with "marco", "pelourinho" or "cruzeiro" (also "Um marco…" and plurals)
small_monument = (places["category"] == "culture") & places["name"].str.contains(
    r"^(?:um )?(?:marcos?|pelourinhos?|cruzeiros?)\b", case=False, regex=True)

places.loc[small_monument, "quality_marker"] = False
print(small_monument.sum(), "small monuments are no longer quality markers")
print("Quality markers now:", places["quality_marker"].sum())

546 small monuments are no longer quality markers
Quality markers now: 4327


---
## 4. 🧀 Typical food of each region

Two lists:
1. **Food products** with a protected name (DOP / IGP): cheese, ham, honey…
2. **Traditional dishes** from the contest *7 Maravilhas da Gastronomia*

### 4.1 Food products (DOP / IGP)

The products don't have a municipality, but the **region is usually in the name**:
*Queijo Serra da Estrela*, *Alheira de Mirandela*, *Ovos Moles de Aveiro*…

We look for **key words** in the name. The order matters: the first word found wins.

In [434]:
food = pd.read_csv(RAW + "dop_igp_products_portugal.csv")

# key word -> region (NUTS II 2013, same as INE)
FOOD_WORDS = [
    ("Ribatejo Norte", "Centro"),
    ("Açores", "Açores"), ("São Miguel", "Açores"), ("S. Miguel", "Açores"), ("Pico", "Açores"),
    ("São Jorge", "Açores"), ("Santa Maria", "Açores"),
    ("Madeira", "Madeira"),
    ("Algarve", "Algarve"), ("Tavira", "Algarve"), ("Monchique", "Algarve"), ("Aljezur", "Algarve"),
    ("Azeitão", "Lisboa"), ("Palmela", "Lisboa"),
    ("Alentej", "Alentejo"), ("Portalegre", "Alentejo"), ("Estremoz", "Alentejo"), ("Évora", "Alentejo"),
    ("Elvas", "Alentejo"), ("Barrancos", "Alentejo"), ("Beja", "Alentejo"), ("Moura", "Alentejo"),
    ("Serpa", "Alentejo"), ("Nisa", "Alentejo"), ("Mertolenga", "Alentejo"), ("Montemor", "Alentejo"),
    ("Santana da Serra", "Alentejo"), ("Tolosa", "Alentejo"), ("Charneca", "Alentejo"),
    ("Ribatejo", "Alentejo"), ("Lezírias", "Alentejo"),
    ("Trás-os-Montes", "Norte"), ("Transmontan", "Norte"), ("Barroso", "Norte"), ("Barrosã", "Norte"),
    ("Vinhais", "Norte"), ("Mirand", "Norte"), ("Bragan", "Norte"), ("Terrincho", "Norte"),
    ("Freixo", "Norte"), ("Douro", "Norte"), ("Padrela", "Norte"), ("Terra Fria", "Norte"),
    ("Terra Quente", "Norte"), ("Minho", "Norte"), ("Peneda", "Norte"), ("Maronesa", "Norte"),
    ("Montezinho", "Norte"), ("Chaves", "Norte"), ("Arouquesa", "Norte"), ("Feira", "Norte"),
    ("Beira", "Centro"), ("Estrela", "Centro"), ("Lousã", "Centro"), ("Rabaçal", "Centro"),
    ("Aveiro", "Centro"), ("Tentúgal", "Centro"), ("Alcobaça", "Centro"), ("Oeste", "Centro"),
    ("Esmolfe", "Centro"), ("Lafões", "Centro"), ("Lapa", "Centro"), ("Marinhoa", "Centro"),
    ("Gralheira", "Centro"),
]

def food_region(product):
    for word, region in FOOD_WORDS:
        if word.lower() in product.lower():
            return region
    return None

food["region"] = food["product"].map(food_region)

print("Without region:", food["region"].isna().sum())
food["region"].value_counts()

Without region: 0


region
Alentejo    47
Norte       42
Centro      26
Açores       7
Algarve      4
Lisboa       2
Madeira      1
Name: count, dtype: int64

In [435]:
# Products where no key word was found (check them!)
food[food["region"].isna()]

,label,category,product,region


### 4.2 Traditional dishes (7 Maravilhas da Gastronomia, 2011)

70 traditional dishes chosen by experts. The column `result` says if the dish **won** (*Vencedor*), was a **finalist** or only a **candidate**.

The regions use old names (*Beira Litoral*, *Entre Douro e Minho*…), so we translate them to our 7 regions.

In [436]:
dishes = pd.read_csv(RAW + "traditional_dishes_7_maravilhas_2011.csv")

# Old region name -> our region
OLD_REGIONS = {
    "Trás-os-Montes e Alto Douro": "Norte", "Entre Douro e Minho": "Norte",
    "Beiras": "Centro", "Beira Litoral": "Centro", "Beira Interior": "Centro",
    "Região de Lisboa": "Lisboa", "Lisboa e Vale do Tejo": "Lisboa", "Lisboa e Setúbal": "Lisboa",
    "Alentejo": "Alentejo", "Alentejo e Algarve": "Alentejo",
    "Algarve": "Algarve", "Açores": "Açores", "Madeira": "Madeira",
}
dishes["region"] = dishes["region"].map(OLD_REGIONS)
dishes["result"] = dishes["result"].replace({"Vencedor": "Winner", "Finalista": "Finalist", "Concorrente": "Candidate"})

print("Dishes without region:", dishes["region"].isna().sum())

Dishes without region: 34


Some dishes have no region in the list. For the ones with a **well-known origin** we add it by hand (e.g. *Bolo do Caco* → Madeira, *Cozido das Furnas* → Açores).
Dishes eaten all over the country (like *Cozido à portuguesa*) stay without region.

In [437]:
DISH_REGION_BY_HAND = {
    "Bôla de Lamego": "Norte", "Bolo do Caco": "Madeira", "Muxama de atum": "Algarve",
    "Espargos com ovos": "Alentejo", "Pezinhos de Coentrada": "Alentejo", "Sopa de castanhas": "Norte",
    "Arroz de lampreia": "Norte", "Bacalhau à Braz": "Lisboa", "Bacalhau à Zé do Pipo": "Norte",
    "Bife de atum à Madeirense": "Madeira", "Camarão da Costa da Figueira": "Centro", "Cracas cozidas": "Açores",
    "Alcatra da Ilha Terceira": "Açores", "Alheira de Mirandela com grelos salteados (IG)": "Norte",
    "Cozido das Furnas": "Açores", "Espetada de carne de vaca em espeto de pau de louro": "Madeira",
    "Posta mirandesa": "Norte", "Perdiz à Algarvia": "Algarve", "Dom Rodrigo": "Algarve",
    "Encharcada do Convento de Santa Clara": "Alentejo", "Ovos moles de Aveiro (IGP)": "Centro",
}
dishes["region"] = dishes["region"].fillna(dishes["dish"].map(DISH_REGION_BY_HAND))

print("Dishes without region now:", dishes["region"].isna().sum())
dishes.groupby("region")["dish"].count().sort_values(ascending=False)

Dishes without region now: 13


region
Norte       11
Alentejo    10
Centro       9
Lisboa       9
Açores       7
Madeira      6
Algarve      5
Name: dish, dtype: int64

In [438]:
# All the dishes of each region
for region, group in dishes.groupby("region"):
    print(f"{region} ({len(group)}): " + ", ".join(group["dish"]))


Alentejo (10): Espargos com ovos, Pezinhos de Coentrada, Presunto de Barrancos (DOP), Açorda à Alentejana, Caspacho com carapaus fritos, Sopa de cação, Migas alentejanas, Encharcada do Convento de Santa Clara, Pão de rala, Sericaia ou Cericaia ou Sericá
Algarve (5): Muxama de atum, Percebes de Aljezur, Xarém com conquilhas, Perdiz à Algarvia, Dom Rodrigo
Açores (7): Sopas do Espírito Santo, Polvo assado no forno, Cracas cozidas, Lapas grelhadas dos Açores, Alcatra da Ilha Terceira, Cozido das Furnas, Ananás dos Açores (DOP)
Centro (9): Queijo Serra da Estrela (DOP), Sopa de peixe da Figueira, Camarão da Costa da Figueira, Chanfana, Leitão da Bairrada, Coelho à Caçador, Perdiz de escabeche, Ovos moles de Aveiro (IGP), Pastéis de Tentúgal (IG)
Lisboa (9): Pastel de bacalhau, Sopa da pedra, Bacalhau à Braz, Sardinha assada, Amêijoas à Bulhão Pato, Arroz de marisco, Mariscada de Sesimbra, Ostras do Sado, Pastel de Belém
Madeira (6): Bolo do Caco, Lapas da Madeira, Bife de atum à Madeirense

---
## 5. One row per municipality 🗺️

We put together, for each municipality:
- the **overnight stays in 2025**
- **how many places** it has of each category
- **how many quality markers** it has

This table will help us find the **hidden gems** (Q5): many quality markers, few tourists.

In [439]:
# Stays in 2025 (the nationality file has full years)
stays_2025 = stays_nat[stays_nat["year"] == 2025].pivot_table(
    index="municipality", columns="country", values="overnight_stays", aggfunc="sum")
summary = pd.DataFrame({"stays_2025": stays_2025["Total"]})

# Number of places per category
counts = places.pivot_table(index="municipality", columns="category", values="name", aggfunc="count", fill_value=0)
counts.columns = ["n_" + c for c in counts.columns]

# Number of quality markers
quality = places[places["quality_marker"] == True].groupby("municipality")["name"].count().rename("n_quality_markers")

municipality_table = (municipalities.set_index("municipality")
                      .join(summary).join(counts).join(quality)
                      .reset_index())

# Municipalities without places get 0 (not empty)
count_columns = list(counts.columns) + ["n_quality_markers"]
municipality_table[count_columns] = municipality_table[count_columns].fillna(0).astype(int)

municipality_table.sort_values("n_quality_markers", ascending=False).head(10)

,municipality,municipality_code,region,stays_2025,n_beach,n_countryside,n_culture,n_nature,n_viewpoint,n_quality_markers
140,Lisboa,1701106,Lisboa,15964064.0,0,0,454,5,56,352
71,Porto,11A1312,Norte,6611174.0,4,0,193,0,24,125
11,Évora,1870705,Alentejo,739187.0,0,0,101,0,2,88
144,Sintra,1701111,Lisboa,744116.0,5,0,86,1,18,75
159,Braga,1120303,Norte,689063.0,2,0,87,1,6,66
188,Coimbra,16E0603,Centro,699087.0,2,0,81,1,5,63
139,Cascais,1701105,Lisboa,1487073.0,7,0,64,3,2,58
60,Ponta Delgada,2004203,Açores,1560205.0,5,0,46,10,38,54
40,Angra do Heroísmo,2004301,Açores,359847.0,6,0,44,11,23,53
2,Funchal,3003103,Madeira,6384904.0,5,0,68,2,52,50


---
## 6. Save the clean files 💾

In [440]:
stays_month.to_csv(CLEAN + "stays_month.csv", index=False)
stays_nat.to_csv(CLEAN + "stays_nationality.csv", index=False)
places.to_csv(CLEAN + "places.csv", index=False)
food.drop(columns="label").to_csv(CLEAN + "food_products.csv", index=False)
dishes.to_csv(CLEAN + "traditional_dishes.csv", index=False)
municipality_table.to_csv(CLEAN + "municipalities.csv", index=False)

print("Saved ✅")

Saved ✅


---
## ✅ Summary

**What we did**
- Kept only the **municipalities** in the INE files (no double counting) and added the region
- Fixed names so all sources use the **same municipality names** as INE
- Gave a municipality to places with only coordinates (closest known place, ~90% right)
- Added **traditional dishes** and **DOP / IGP products** for the "You must try" tips of each package
- Put all the places in **one table** and created **one row per municipality**

**Next notebook (02):** statistics + answers to the 5 questions.